# Curvilinear-grid NetCDF files

Grids with 2-D latitude/longitude coordinate arrays (no single affine transform). These support inspection, plotting (via pcolormesh), retrieval, and variable mutation, and polygon crop (by masking on the 2-D coordinates — see below).

In [ ]:
%matplotlib inline
import tempfile
from pathlib import Path

import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon

from pyramids.feature import FeatureCollection
from pyramids.netcdf import ColorOpts, NetCDF

DATA = Path('../../../../examples/data/netcdf/samples')

## `cf__8v__1d3-2d3-3d1-4d1__curv-stag.nc`

ROMS ocean model output on a curvilinear, staggered (Arakawa-C) grid.

**Open the file and inspect the container**

In [ ]:
nc = NetCDF.read_file(DATA / 'cf__8v__1d3-2d3-3d1-4d1__curv-stag.nc')
nc

**Dimensions and variables**

In [ ]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

**Global attributes**

In [ ]:
nc.global_attributes

**Plot the variable** (a 2-D slice is auto-selected for >2-D variables)

In [ ]:
# clamp the colour scale to the open-ocean range: the coastal freshwater plumes
# pull the default min down to ~9 PSU, which flattens the offshore salinity
# structure into near-uniform blue
glyph = nc.plot(variable='salt', colour=ColorOpts(vmin=30, vmax=37))
# the glyph carries the variable's CRS (issue #630), so the Natural Earth coastline
# overlay lines up with this curvilinear lon/lat grid without restating crs=
glyph.add_features("coastline", "50m", zorder=5)

**Retrieve the underlying data**

In [ ]:
var = nc.get_variable('salt')
data = var.read_array()
print('shape:', data.shape)
print(
    'min / mean / max:',
    float(np.nanmin(data)),
    float(np.nanmean(data)),
    float(np.nanmax(data)),
)

**Add a variable** — derive a 2-D field and append it as a new variable

In [ ]:
work = Path(tempfile.mkdtemp())
slice2d = data[tuple(0 for _ in range(data.ndim - 2))]
NetCDF.create_from_array(
    arr=slice2d,
    geo=(0.0, 1.0, 0.0, 0.0, 0.0, -1.0),
    epsg=var.epsg or 4326,
    variable_name='salt_slice0',
    path=str(work / 'derived.nc'),
)
nc.add_variable(NetCDF.read_file(str(work / 'derived.nc')), 'salt_slice0')
print('variables after add:', nc.variable_names)

**Remove a variable**

In [ ]:
nc.remove_variable('salt_slice0')
print('variables after remove:', nc.variable_names)

**Crop with a polygon** — this grid is **curvilinear** (2-D `lat_rho`/`lon_rho`), so there is no single affine transform to clip against. `crop` therefore masks each cell whose centre falls outside the polygon to no-data and trims to the bounding window, **keeping the 2-D coordinates** so the result stays curvilinear and still plots on its real geometry.

In [ ]:
# a box over the eastern Gulf shelf (lon -91..-88, lat 28..30.5)
aoi = FeatureCollection(
    gpd.GeoDataFrame(
        geometry=[Polygon([(-91, 28), (-88, 28), (-88, 30.5), (-91, 30.5)])], crs=4326
    )
)
salt = nc.get_variable('salt')
cropped = salt.crop(aoi)
print('cropped shape:', cropped.shape)
glyph = cropped.plot()
# overlay the coastline so the cropped curvilinear patch is easy to place
glyph.add_features("coastline", "50m", zorder=5)

**Save the container to a new NetCDF file**

In [ ]:
out = work / 'saved.nc'
nc.to_file(out)
print('saved container to', out.name, '->', out.exists())

## `none__4v__1d1-2d2-3d1__curv.nc`

WRF/RASM air temperature on a curvilinear grid (2-D xc/yc coordinates).

**Open the file and inspect the container**

In [ ]:
nc = NetCDF.read_file(DATA / 'none__4v__1d1-2d2-3d1__curv.nc')
nc

**Dimensions and variables**

In [ ]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

**Global attributes**

In [ ]:
nc.global_attributes

**Plot the variable** (a 2-D slice is auto-selected for >2-D variables)

In [ ]:
# This RASM grid wraps 0–360° longitude: its easternmost columns cross the 0/360
# "seam", which a flat pcolormesh can't draw (a seam-spanning cell smears across the
# whole width). Restrict to the seam-free western columns and plot on the variable's
# own 2-D curvilinear (xc, yc) coordinates.
xc = np.asarray(nc.get_variable('xc').read_array(), float)
yc = np.asarray(nc.get_variable('yc').read_array(), float)
tair = nc.get_variable('Tair').read_array()[0].astype(float)  # first time-step
tair[tair > 1e30] = (
    np.nan
)  # mask the fill value so it doesn't dominate the colour scale
lon_jump = np.abs(np.diff(np.where(xc > 1e30, np.nan, xc), axis=1)) > 100
cols = slice(0, int(np.argmax(lon_jump.any(axis=0))) or xc.shape[1])
patch = NetCDF.create_from_array(
    tair[:, cols], geo=(0, 1, 0, tair.shape[0], 0, -1), epsg=4326, variable_name='Tair'
)
glyph = patch.plot(variable='Tair', coords=(xc[:, cols], yc[:, cols]))
glyph.add_features("coastline", "50m", zorder=5)

**Retrieve the underlying data**

In [ ]:
var = nc.get_variable('Tair')
data = var.read_array()
print('shape:', data.shape)
print(
    'min / mean / max:',
    float(np.nanmin(data)),
    float(np.nanmean(data)),
    float(np.nanmax(data)),
)

**Add a variable** — derive a 2-D field and append it as a new variable

In [ ]:
work = Path(tempfile.mkdtemp())
slice2d = data[tuple(0 for _ in range(data.ndim - 2))]
NetCDF.create_from_array(
    arr=slice2d,
    geo=(0.0, 1.0, 0.0, 0.0, 0.0, -1.0),
    epsg=var.epsg or 4326,
    variable_name='Tair_slice0',
    path=str(work / 'derived.nc'),
)
nc.add_variable(NetCDF.read_file(str(work / 'derived.nc')), 'Tair_slice0')
print('variables after add:', nc.variable_names)

**Remove a variable**

In [ ]:
nc.remove_variable('Tair_slice0')
print('variables after remove:', nc.variable_names)

> **Polygon crop** — like the ROMS example above, this curvilinear grid is cropped by masking on its 2-D coordinates (cells whose centre is outside the polygon become no-data, then the grid is trimmed to the bounding window). The result keeps its 2-D coordinates and stays curvilinear.

**Save the container to a new NetCDF file**

In [ ]:
out = work / 'saved.nc'
nc.to_file(out)
print('saved container to', out.name, '->', out.exists())

## `none__5v__1d2-2d2-3d1__curv.nc`

Satellite IMAGE product with 2-D latitude/longitude and string metadata variables.

**Open the file and inspect the container**

In [ ]:
nc = NetCDF.read_file(DATA / 'none__5v__1d2-2d2-3d1__curv.nc')
nc

**Dimensions and variables**

In [ ]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

**Global attributes**

In [ ]:
nc.global_attributes

**Plot the variable** (a 2-D slice is auto-selected for >2-D variables)

In [ ]:
nc.plot(variable='data')

**Retrieve the underlying data**

In [ ]:
var = nc.get_variable('data')
data = var.read_array()
print('shape:', data.shape)
print(
    'min / mean / max:',
    float(np.nanmin(data)),
    float(np.nanmean(data)),
    float(np.nanmax(data)),
)

**Add a variable** — derive a 2-D field and append it as a new variable

In [ ]:
work = Path(tempfile.mkdtemp())
slice2d = data[tuple(0 for _ in range(data.ndim - 2))]
NetCDF.create_from_array(
    arr=slice2d,
    geo=(0.0, 1.0, 0.0, 0.0, 0.0, -1.0),
    epsg=var.epsg or 4326,
    variable_name='data_slice0',
    path=str(work / 'derived.nc'),
)
nc.add_variable(NetCDF.read_file(str(work / 'derived.nc')), 'data_slice0')
print('variables after add:', nc.variable_names)

**Remove a variable**

In [ ]:
nc.remove_variable('data_slice0')
print('variables after remove:', nc.variable_names)

> **Polygon crop** — like the ROMS example above, this curvilinear grid is cropped by masking on its 2-D coordinates (cells whose centre is outside the polygon become no-data, then the grid is trimmed to the bounding window). The result keeps its 2-D coordinates and stays curvilinear.

**Save the container to a new NetCDF file**

In [ ]:
out = work / 'saved.nc'
nc.to_file(out)
print('saved container to', out.name, '->', out.exists())